# 公式テスト t5 — Mean Square Displacement (MSD)

> **出典（E-Cell4 公式）**: Tests / MSD — https://ecell4.e-cell.org/tests/MSD.html
>
> 単分子空間ソルバ（eGFRD / Spatiocyte）で拡散する 1 分子の**平均二乗変位** MSD が、理論 `6Dt`（3次元）に一致するかの検証。
>
> ⚠️ **本ノートは自動実行していません**：軌道を追う単分子シミュレーション（`FixedIntervalTrajectoryObserver`）は
> 空間・重い。公式コードは忠実に掲載。手元で実行を。

In [ ]:
# 公式コード（そのまま）。※単分子・軌道追跡・重い
import matplotlib.pyplot as plt
import numpy
from ecell4.prelude import *

radius, D = 0.005, 1
m = NetworkModel()
m.add_species_attribute(Species('A', radius, D))

rng = GSLRandomNumberGenerator(); rng.seed(0)
duration = 3.0; N = 50
obs = FixedIntervalTrajectoryObserver(0.01)

ret1 = ensemble_simulations(duration, ndiv=1, model=m, y0={'A': N}, observers=obs,
                            solver=('egfrd', Integer3(4, 4, 4)), repeat=10)
ret2 = ensemble_simulations(duration, ndiv=1, model=m, y0={'A': N}, observers=obs,
                            solver=('spatiocyte', radius), repeat=10)

def test_mean_square_displacement(ret):
    t = numpy.array(ret[0].observers[1].t()); sd = []
    for ret_ in ret:
        for data in ret_.observers[1].data():
            sd.append(numpy.array([length_sq(pos - data[0]) for pos in data]))
    return (t, numpy.average(sd, axis=0), numpy.std(sd, axis=0))

t, mean1, std1 = test_mean_square_displacement(ret1)
t, mean2, std2 = test_mean_square_displacement(ret2)

plt.plot(t, 6 * D * t, 'k-', label='Expected (6 D t)')
plt.errorbar(t[::30], mean1[::30], yerr=std1[::30], fmt='go', label='eGFRD')
plt.errorbar(t[::30], mean2[::30], yerr=std2[::30], fmt='ro', label='Spatiocyte')
plt.xlabel('Time'); plt.ylabel('Mean Square Displacement'); plt.legend(loc='best', shadow=True); plt.show()

## 説明

自由拡散する粒子は、時間 `t` 後の平均二乗変位が **MSD = 6Dt**（3 次元, D=拡散係数）になる（アインシュタインの関係）。
eGFRD と Spatiocyte の両単分子ソルバが、この理論直線にのることを確認する**拡散の妥当性テスト**。

**要点**: 反応を含まない純粋な拡散でも、空間ソルバが物理的に正しく粒子を動かしているかを MSD で検証できる。
`FixedIntervalTrajectoryObserver` が 1 粒子ごとの軌跡を記録し、`length_sq(pos - data[0])` で初期位置からの変位²を測る。